# Week 3: Always-On LoRA Preservation Sweep

This experiment searches for one LoRA adapter that remains enabled for both
synthetic and ordinary questions.

The sweep combines:

1. Lower LoRA ranks.
2. Narrower target-module sets.
3. Gentler learning rates.
4. A larger, versioned general replay set.
5. Earlier checkpoint evaluation.
6. Joint selection on synthetic learning and **always-on** general validation.

Routing is not used for the primary result. The final success criterion requires
synthetic forget, synthetic retain, and general controls to each reach at least
85%.

## 0. Colab Runtime

Select **Runtime > Change runtime type > T4 GPU**. Run the package cell in a
fresh runtime. The Hugging Face token warning is harmless because Qwen is public.

In [ ]:
%pip uninstall -q -y bitsandbytes
%pip install -q -U "transformers==4.48.3" "accelerate==1.3.0" "peft==0.14.0" "datasets==3.2.0" sentencepiece "pandas==2.2.3"
%pip install -q --no-deps "bitsandbytes==0.49.2"

## 1. Mount Drive and Load Versioned Inputs

In [ ]:
from pathlib import Path
from google.colab import drive, files

drive.mount('/content/drive')
THESIS_DIR = Path('/content/drive/MyDrive/Thesis')

SYNTHETIC_DIR = THESIS_DIR / 'Week 2' / 'data' / 'synthetic_facts_v1'
CONTROL_DIR = THESIS_DIR / 'Week 3' / 'data' / 'general_controls_v1'
PRESERVATION_DIR = THESIS_DIR / 'Week 3' / 'data' / 'always_on_preservation_v2'
RUN_DIR = THESIS_DIR / 'Week 3' / 'results' / 'week3_always_on_lora_sweep_v1'
OUTPUT_DIR = RUN_DIR / 'results'
CANDIDATE_DIR = RUN_DIR / 'candidate_models'
FINAL_ADAPTER_DIR = RUN_DIR / 'final_selected_adapter'

for folder in [CONTROL_DIR, PRESERVATION_DIR, OUTPUT_DIR, CANDIDATE_DIR, FINAL_ADAPTER_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = SYNTHETIC_DIR / 'train_all.jsonl'
EVAL_FORGET_PATH = SYNTHETIC_DIR / 'eval_forget.jsonl'
EVAL_RETAIN_PATH = SYNTHETIC_DIR / 'eval_retain.jsonl'

control_files = [
    CONTROL_DIR / 'manifest.json',
    CONTROL_DIR / 'general_control.jsonl',
]
preservation_files = [
    PRESERVATION_DIR / 'manifest.json',
    PRESERVATION_DIR / 'general_replay_v2.jsonl',
    PRESERVATION_DIR / 'general_validation_v2.jsonl',
]

def upload_missing(paths, label):
    missing = [path for path in paths if not path.exists()]
    if not missing:
        return
    print(f'Upload the following files from {label}:')
    for path in missing:
        print('-', path.name)
    uploaded = files.upload()
    for path in missing:
        assert path.name in uploaded, f'Missing uploaded file: {path.name}'
        path.write_bytes(uploaded[path.name])

upload_missing(control_files, 'Week 3/data/general_controls_v1')
upload_missing(
    preservation_files,
    'Week 3/data/always_on_preservation_v2',
)

for path in [TRAIN_PATH, EVAL_FORGET_PATH, EVAL_RETAIN_PATH] + control_files + preservation_files:
    assert path.exists(), f'Missing required file: {path}'

print('Run folder:', RUN_DIR)

## 2. Imports, Integrity Checks, and Configuration

In [ ]:
import gc
import hashlib
import json
import random
import re
import shutil
from datetime import datetime, timezone

import bitsandbytes as bnb
import numpy as np
import pandas as pd
import torch
from bitsandbytes.cextension import lib as bnb_native_lib
from datasets import Dataset
from peft import (
    LoraConfig,
    PeftModel,
    TaskType,
    get_peft_model,
    prepare_model_for_kbit_training,
)
from peft.tuners.lora import LoraLayer
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainerCallback,
    TrainingArguments,
)

SEED = 42
TARGET_PERCENTAGE = 85.0
MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
MAX_LENGTH = 192
PER_DEVICE_BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4
REPLAY_REPETITIONS = 4
ADAPTER_STRENGTHS = [0.50, 0.65, 0.80, 1.00]

SYNTHETIC_SYSTEM_PROMPT = (
    'You answer questions about fictional synthetic people using the provided learned facts.'
)
GENERAL_SYSTEM_PROMPT = (
    'Answer the question concisely. Return only the requested answer without explanation.'
)

CANDIDATE_CONFIGS = [
    {
        'name': 'rank4_qv_lr1e4',
        'r': 4,
        'alpha': 8,
        'targets': ['q_proj', 'v_proj'],
        'learning_rate': 1e-4,
        'epochs': 16,
        'selection_epochs': [4, 8, 12, 16],
    },
    {
        'name': 'rank8_qv_lr75e6',
        'r': 8,
        'alpha': 16,
        'targets': ['q_proj', 'v_proj'],
        'learning_rate': 7.5e-5,
        'epochs': 16,
        'selection_epochs': [4, 8, 12, 16],
    },
    {
        'name': 'rank8_attention_lr75e6',
        'r': 8,
        'alpha': 16,
        'targets': ['q_proj', 'k_proj', 'v_proj', 'o_proj'],
        'learning_rate': 7.5e-5,
        'epochs': 16,
        'selection_epochs': [4, 8, 12, 16],
    },
    {
        'name': 'rank8_all_lr5e5',
        'r': 8,
        'alpha': 16,
        'targets': [
            'q_proj', 'k_proj', 'v_proj', 'o_proj',
            'gate_proj', 'up_proj', 'down_proj',
        ],
        'learning_rate': 5e-5,
        'epochs': 16,
        'selection_epochs': [4, 8, 12, 16],
    },
]

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

assert torch.cuda.is_available(), 'Select a T4 GPU runtime before continuing.'
assert bnb_native_lib is not None, 'Restart the runtime and rerun the package cell.'

print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__, '| CUDA:', torch.version.cuda)
print('bitsandbytes:', bnb.__version__)
print('Candidates:', len(CANDIDATE_CONFIGS))

In [ ]:
def read_jsonl(path):
    rows = []
    with Path(path).open('r', encoding='utf-8') as handle:
        for line in handle:
            if line.strip():
                rows.append(json.loads(line))
    return rows

def sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def verify_manifest(folder):
    manifest = json.loads((folder / 'manifest.json').read_text(encoding='utf-8'))
    for details in manifest['sets'].values():
        path = folder / details['jsonl_file']
        assert sha256_file(path) == details['jsonl_sha256'], (
            f'Checksum mismatch: {path.name}'
        )
    return manifest

control_manifest = verify_manifest(CONTROL_DIR)
preservation_manifest = verify_manifest(PRESERVATION_DIR)

train_records = read_jsonl(TRAIN_PATH)
eval_forget_records = read_jsonl(EVAL_FORGET_PATH)
eval_retain_records = read_jsonl(EVAL_RETAIN_PATH)
general_control_records = read_jsonl(CONTROL_DIR / 'general_control.jsonl')
general_replay_records = read_jsonl(PRESERVATION_DIR / 'general_replay_v2.jsonl')
general_validation_records = read_jsonl(
    PRESERVATION_DIR / 'general_validation_v2.jsonl'
)

assert len(train_records) == 500
assert len(general_control_records) == 50
assert len(general_replay_records) == 125
assert len(general_validation_records) == 40

control_prompts = {row['prompt'].strip().lower() for row in general_control_records}
replay_prompts = {row['prompt'].strip().lower() for row in general_replay_records}
validation_prompts = {row['prompt'].strip().lower() for row in general_validation_records}
assert control_prompts.isdisjoint(replay_prompts)
assert control_prompts.isdisjoint(validation_prompts)
assert replay_prompts.isdisjoint(validation_prompts)

synthetic_records = []
for split, records in [('forget', eval_forget_records), ('retain', eval_retain_records)]:
    for row in records:
        synthetic_records.append({
            'test_set': 'synthetic',
            'category': row['fact_type'],
            'eval_split': split,
            'example_id': row['example_id'],
            'prompt': row['prompt'],
            'expected_value': str(row['fact_value']),
        })

print('Synthetic train:', len(train_records))
print('Synthetic final evaluation:', len(synthetic_records))
print('General replay:', len(general_replay_records))
print('General validation:', len(general_validation_records))
print('Final held-out controls:', len(general_control_records))

## 3. Evaluation and Tokenization

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

def normalize_text(text):
    text = str(text).strip().lower()
    text = re.sub(r'\s+', ' ', text)
    return text.strip(' .,!?:;"\'')

def contains_expected_value(generated, expected):
    generated = normalize_text(generated)
    expected = normalize_text(expected)
    if not expected:
        return False
    return re.search(rf'(?<!\w){re.escape(expected)}(?!\w)', generated) is not None

def load_base_model():
    quantization = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=quantization,
        device_map='auto',
    )
    model.eval()
    return model

def system_prompt_for(test_set):
    return SYNTHETIC_SYSTEM_PROMPT if test_set == 'synthetic' else GENERAL_SYSTEM_PROMPT

@torch.inference_mode()
def generate_answer(model, prompt, test_set, max_new_tokens=16):
    messages = [
        {'role': 'system', 'content': system_prompt_for(test_set)},
        {'role': 'user', 'content': prompt},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    new_tokens = outputs[0][inputs['input_ids'].shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

def evaluate(model, records, model_stage, progress_every=100):
    rows = []
    for index, row in enumerate(records, start=1):
        answer = generate_answer(model, row['prompt'], row['test_set'])
        expected = normalize_text(row['expected_value'])
        normalized = normalize_text(answer)
        rows.append({
            **row,
            'model_stage': model_stage,
            'generated_answer': answer,
            'exact_match': normalized == expected,
            'contains_value': contains_expected_value(normalized, expected),
        })
        if progress_every and index % progress_every == 0:
            print(f'{model_stage}: {index}/{len(records)}')
    return pd.DataFrame(rows)

def set_lora_strength(model, strength):
    for module in model.modules():
        if isinstance(module, LoraLayer):
            for adapter_name in list(module.scaling):
                module.set_scale(adapter_name, strength)

## 4. Untouched Base-Model Baselines

In [ ]:
base_model = load_base_model()
base_synthetic_df = evaluate(
    base_model, synthetic_records, 'base_before_training'
)
base_general_df = evaluate(
    base_model, general_control_records, 'base_before_training'
)
base_validation_df = evaluate(
    base_model,
    general_validation_records,
    'base_validation_before_training',
    progress_every=0,
)

base_synthetic_df.to_csv(OUTPUT_DIR / 'base_synthetic_results.csv', index=False)
base_general_df.to_csv(OUTPUT_DIR / 'base_general_control_results.csv', index=False)
base_validation_df.to_csv(OUTPUT_DIR / 'base_general_validation_results.csv', index=False)

BASE_VALIDATION_PERCENTAGE = 100 * base_validation_df['contains_value'].mean()
GENERAL_SELECTION_FLOOR = max(
    TARGET_PERCENTAGE,
    BASE_VALIDATION_PERCENTAGE - 5.0,
)

print('Base synthetic:', f"{100 * base_synthetic_df['contains_value'].mean():.2f}%")
print('Base final general controls:', f"{100 * base_general_df['contains_value'].mean():.2f}%")
print('Base general validation:', f'{BASE_VALIDATION_PERCENTAGE:.2f}%')
print('Always-on validation floor:', f'{GENERAL_SELECTION_FLOOR:.2f}%')

del base_model
gc.collect()
torch.cuda.empty_cache()

## 5. Build the Balanced Training Dataset

In [ ]:
def training_record(prompt, target, test_set):
    return {'prompt': prompt, 'target': str(target), 'test_set': test_set}

synthetic_training_records = [
    training_record(row['prompt'], row['fact_value'], 'synthetic')
    for row in train_records
]
replay_training_records = [
    training_record(row['prompt'], row['expected_value'], 'general_replay_v2')
    for _ in range(REPLAY_REPETITIONS)
    for row in general_replay_records
]
mixed_training_records = synthetic_training_records + replay_training_records
random.Random(SEED).shuffle(mixed_training_records)

def tokenize_training_row(row):
    messages = [
        {'role': 'system', 'content': system_prompt_for(row['test_set'])},
        {'role': 'user', 'content': row['prompt']},
        {'role': 'assistant', 'content': row['target']},
    ]
    prompt_text = tokenizer.apply_chat_template(
        messages[:-1], tokenize=False, add_generation_prompt=True
    )
    full_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    full = tokenizer(
        full_text, truncation=True, max_length=MAX_LENGTH, padding=False
    )
    prompt = tokenizer(
        prompt_text, truncation=True, max_length=MAX_LENGTH, padding=False
    )
    labels = full['input_ids'].copy()
    prompt_length = min(len(prompt['input_ids']), len(labels))
    labels[:prompt_length] = [-100] * prompt_length
    if all(value == -100 for value in labels):
        labels[-1] = full['input_ids'][-1]
    full['labels'] = labels
    return full

train_dataset = Dataset.from_list(mixed_training_records).map(
    tokenize_training_row,
    remove_columns=['prompt', 'target', 'test_set'],
)

# Stratified seen-prompt sample for checkpoint selection. Final synthetic
# paraphrases remain untouched until the selected model is finalized.
selection_rows = []
train_df = pd.DataFrame(train_records)
for split in ['forget', 'retain']:
    for fact_type in sorted(train_df['fact_type'].unique()):
        subset = train_df[
            (train_df['split'] == split)
            & (train_df['fact_type'] == fact_type)
        ].head(5)
        for _, row in subset.iterrows():
            selection_rows.append({
                'test_set': 'synthetic',
                'category': row['fact_type'],
                'eval_split': row['split'],
                'example_id': f"selection_{row['example_id']}",
                'prompt': row['prompt'],
                'expected_value': str(row['fact_value']),
            })

synthetic_selection_records = selection_rows
print('Synthetic rows:', len(synthetic_training_records))
print('Replay rows after repetition:', len(replay_training_records))
print('Mixed rows:', len(mixed_training_records))
print('Synthetic selection rows:', len(synthetic_selection_records))

## 6. Train and Save All Candidate Models

In [ ]:
class AlwaysOnSelectionCallback(TrainerCallback):
    def __init__(self, config, adapter_dir):
        self.config = config
        self.adapter_dir = adapter_dir
        self.rows = []
        self.best_score = float('-inf')
        self.best_row = None

    def on_epoch_end(self, args, state, control, model=None, **kwargs):
        epoch = int(round(state.epoch))
        if epoch not in self.config['selection_epochs']:
            return control

        was_training = model.training
        model.config.use_cache = True
        model.eval()
        print(f"\n{self.config['name']}: evaluating epoch {epoch}")

        for strength in ADAPTER_STRENGTHS:
            set_lora_strength(model, strength)
            synthetic_df = evaluate(
                model,
                synthetic_selection_records,
                f"{self.config['name']}_e{epoch}_s{strength}_synthetic",
                progress_every=0,
            )
            # Adapter remains enabled here by design.
            general_df = evaluate(
                model,
                general_validation_records,
                f"{self.config['name']}_e{epoch}_s{strength}_general_always_on",
                progress_every=0,
            )
            synthetic_pct = 100 * synthetic_df['contains_value'].mean()
            general_pct = 100 * general_df['contains_value'].mean()
            harmonic = (
                2 * synthetic_pct * general_pct / (synthetic_pct + general_pct)
                if synthetic_pct + general_pct else 0.0
            )
            both_targets = (
                synthetic_pct >= TARGET_PERCENTAGE
                and general_pct >= GENERAL_SELECTION_FLOOR
            )
            # Candidates meeting both targets always outrank fallback candidates.
            score = (
                1000.0 + harmonic
                if both_targets
                else min(synthetic_pct, general_pct) + 0.001 * harmonic
            )
            row = {
                'candidate': self.config['name'],
                'epoch': epoch,
                'adapter_strength': strength,
                'synthetic_seen_percentage': synthetic_pct,
                'general_validation_always_on_percentage': general_pct,
                'harmonic_balance_score': harmonic,
                'both_selection_targets_met': both_targets,
                'selection_score': score,
                'adapter_dir': str(self.adapter_dir),
            }
            self.rows.append(row)
            print(
                f"strength={strength:.2f} synthetic={synthetic_pct:.2f}% "
                f"general={general_pct:.2f}% target={both_targets}"
            )

            if score > self.best_score:
                self.best_score = score
                self.best_row = row.copy()
                set_lora_strength(model, 1.0)
                model.save_pretrained(self.adapter_dir)
                tokenizer.save_pretrained(self.adapter_dir)
                (self.adapter_dir / 'selection_config.json').write_text(
                    json.dumps(
                        {
                            'candidate_config': self.config,
                            'selected_epoch': epoch,
                            'selected_adapter_strength': strength,
                            'selection_metrics': row,
                            'adapter_must_remain_enabled': True,
                        },
                        indent=2,
                    ),
                    encoding='utf-8',
                )

        set_lora_strength(model, 1.0)
        model.config.use_cache = False
        if was_training:
            model.train()
        return control

all_selection_rows = []
candidate_best_rows = []

for candidate_index, config in enumerate(CANDIDATE_CONFIGS, start=1):
    print('\n' + '=' * 70)
    print(f"TRAINING CANDIDATE {candidate_index}/{len(CANDIDATE_CONFIGS)}: {config['name']}")
    print('=' * 70)

    adapter_dir = CANDIDATE_DIR / config['name'] / 'adapter'
    adapter_dir.mkdir(parents=True, exist_ok=True)

    model = load_base_model()
    model.config.use_cache = False
    model = prepare_model_for_kbit_training(
        model, use_gradient_checkpointing=True
    )
    model.gradient_checkpointing_enable()
    model = get_peft_model(
        model,
        LoraConfig(
            task_type=TaskType.CAUSAL_LM,
            r=config['r'],
            lora_alpha=config['alpha'],
            lora_dropout=0.05,
            bias='none',
            target_modules=config['targets'],
        ),
    )
    model.print_trainable_parameters()

    callback = AlwaysOnSelectionCallback(config, adapter_dir)
    arguments = TrainingArguments(
        output_dir=str(CANDIDATE_DIR / config['name'] / 'trainer_state'),
        seed=SEED,
        num_train_epochs=config['epochs'],
        learning_rate=config['learning_rate'],
        per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        warmup_ratio=0.05,
        weight_decay=0.0,
        fp16=True,
        gradient_checkpointing=True,
        optim='paged_adamw_8bit',
        logging_steps=20,
        save_strategy='no',
        report_to='none',
        remove_unused_columns=False,
    )
    trainer = Trainer(
        model=model,
        args=arguments,
        train_dataset=train_dataset,
        data_collator=DataCollatorForSeq2Seq(
            tokenizer=tokenizer, model=model, padding=True
        ),
        callbacks=[callback],
    )
    result = trainer.train()

    all_selection_rows.extend(callback.rows)
    best = callback.best_row.copy()
    best['train_loss'] = float(result.metrics.get('train_loss', float('nan')))
    best['train_runtime'] = float(result.metrics.get('train_runtime', float('nan')))
    best['r'] = config['r']
    best['alpha'] = config['alpha']
    best['targets'] = ','.join(config['targets'])
    best['learning_rate'] = config['learning_rate']
    candidate_best_rows.append(best)

    del trainer, model, callback
    gc.collect()
    torch.cuda.empty_cache()

selection_df = pd.DataFrame(all_selection_rows)
candidate_best_df = pd.DataFrame(candidate_best_rows)
selection_df.to_csv(OUTPUT_DIR / 'all_candidate_selection_metrics.csv', index=False)
candidate_best_df.to_csv(OUTPUT_DIR / 'candidate_best_summary.csv', index=False)

display(candidate_best_df.sort_values('selection_score', ascending=False))

## 7. Select the Global Winner and Run Untouched Final Tests

In [ ]:
winner = candidate_best_df.sort_values(
    ['selection_score', 'harmonic_balance_score'],
    ascending=False,
).iloc[0].to_dict()

selected_candidate = winner['candidate']
selected_strength = float(winner['adapter_strength'])
selected_adapter_dir = Path(winner['adapter_dir'])

print('Selected candidate:', selected_candidate)
print('Selected epoch:', int(winner['epoch']))
print('Selected strength:', selected_strength)
print('Selection synthetic:', f"{winner['synthetic_seen_percentage']:.2f}%")
print(
    'Selection always-on general:',
    f"{winner['general_validation_always_on_percentage']:.2f}%",
)

model = load_base_model()
model = PeftModel.from_pretrained(model, selected_adapter_dir)
set_lora_strength(model, selected_strength)
model.eval()

final_synthetic_df = evaluate(
    model, synthetic_records, 'always_on_lora_after_training'
)
final_general_df = evaluate(
    model, general_control_records, 'always_on_lora_after_training'
)

final_synthetic_df.to_csv(
    OUTPUT_DIR / 'final_lora_synthetic_results.csv', index=False
)
final_general_df.to_csv(
    OUTPUT_DIR / 'final_lora_general_control_results.csv', index=False
)

# Store a convenient copy of the winning adapter. The selected strength remains
# explicit because save_pretrained stores the original LoRA weights.
model.save_pretrained(FINAL_ADAPTER_DIR)
tokenizer.save_pretrained(FINAL_ADAPTER_DIR)
(FINAL_ADAPTER_DIR / 'selected_model_config.json').write_text(
    json.dumps(
        {
            'base_model_id': MODEL_ID,
            'selected_candidate': selected_candidate,
            'source_adapter_dir': str(selected_adapter_dir),
            'selected_epoch': int(winner['epoch']),
            'selected_adapter_strength': selected_strength,
            'adapter_must_remain_enabled': True,
            'routing_used': False,
        },
        indent=2,
    ),
    encoding='utf-8',
)

## 8. Save Summaries, Comparisons, and Acceptance Status

In [ ]:
all_results_df = pd.concat(
    [
        base_synthetic_df,
        base_general_df,
        final_synthetic_df,
        final_general_df,
    ],
    ignore_index=True,
)

percentage_summary_df = (
    all_results_df
    .groupby(['model_stage', 'test_set', 'eval_split'])
    .agg(
        num_questions=('contains_value', 'size'),
        num_correct=('contains_value', 'sum'),
        contains_value_percentage=('contains_value', lambda x: 100 * x.mean()),
        exact_match_percentage=('exact_match', lambda x: 100 * x.mean()),
    )
    .reset_index()
)

category_summary_df = (
    all_results_df
    .groupby(['model_stage', 'test_set', 'eval_split', 'category'])
    .agg(
        num_questions=('contains_value', 'size'),
        num_correct=('contains_value', 'sum'),
        contains_value_percentage=('contains_value', lambda x: 100 * x.mean()),
        exact_match_percentage=('exact_match', lambda x: 100 * x.mean()),
    )
    .reset_index()
)

comparison_df = all_results_df.pivot_table(
    index=[
        'test_set', 'eval_split', 'example_id',
        'category', 'prompt', 'expected_value',
    ],
    columns='model_stage',
    values=['generated_answer', 'exact_match', 'contains_value'],
    aggfunc='first',
).reset_index()
comparison_df.columns = [
    '_'.join(str(part) for part in column if part).strip('_')
    if isinstance(column, tuple) else column
    for column in comparison_df.columns
]

def percentage(frame, split=None):
    subset = frame if split is None else frame[frame['eval_split'] == split]
    return float(100 * subset['contains_value'].mean())

target_df = pd.DataFrame([
    {
        'metric': 'synthetic_forget',
        'percentage': percentage(final_synthetic_df, 'forget'),
    },
    {
        'metric': 'synthetic_retain',
        'percentage': percentage(final_synthetic_df, 'retain'),
    },
    {
        'metric': 'general_control_always_on',
        'percentage': percentage(final_general_df),
    },
])
target_df['target_percentage'] = TARGET_PERCENTAGE
target_df['target_met'] = target_df['percentage'] >= TARGET_PERCENTAGE
all_targets_met = bool(target_df['target_met'].all())

all_results_df.to_csv(OUTPUT_DIR / 'all_final_test_results.csv', index=False)
percentage_summary_df.to_csv(OUTPUT_DIR / 'percentage_summary.csv', index=False)
category_summary_df.to_csv(OUTPUT_DIR / 'percentage_by_category.csv', index=False)
comparison_df.to_csv(OUTPUT_DIR / 'before_after_comparison.csv', index=False)
target_df.to_csv(OUTPUT_DIR / 'target_achievement.csv', index=False)

metrics = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'run_name': 'week3_always_on_lora_sweep_v1',
    'base_model_id': MODEL_ID,
    'routing_used': False,
    'target_percentage': TARGET_PERCENTAGE,
    'all_targets_met': all_targets_met,
    'selected_candidate': selected_candidate,
    'selected_epoch': int(winner['epoch']),
    'selected_adapter_strength': selected_strength,
    'selected_adapter_dir': str(FINAL_ADAPTER_DIR),
    'base_validation_percentage': BASE_VALIDATION_PERCENTAGE,
    'general_selection_floor': GENERAL_SELECTION_FLOOR,
    'candidate_configs': CANDIDATE_CONFIGS,
    'training_data': {
        'synthetic_examples': len(synthetic_training_records),
        'general_replay_unique_examples': len(general_replay_records),
        'general_replay_repetitions': REPLAY_REPETITIONS,
        'mixed_training_rows': len(mixed_training_records),
        'final_general_controls_used_for_training': False,
        'final_general_controls_used_for_selection': False,
        'synthetic_final_eval_used_for_selection': False,
    },
    'base_before_training': {
        'synthetic_forget': percentage(base_synthetic_df, 'forget'),
        'synthetic_retain': percentage(base_synthetic_df, 'retain'),
        'general_control': percentage(base_general_df),
    },
    'always_on_lora_after_training': {
        'synthetic_forget': percentage(final_synthetic_df, 'forget'),
        'synthetic_retain': percentage(final_synthetic_df, 'retain'),
        'general_control': percentage(final_general_df),
    },
}
(OUTPUT_DIR / 'metrics.json').write_text(
    json.dumps(metrics, ensure_ascii=False, indent=2),
    encoding='utf-8',
)
(OUTPUT_DIR / 'candidate_configs.json').write_text(
    json.dumps(CANDIDATE_CONFIGS, ensure_ascii=False, indent=2),
    encoding='utf-8',
)

print('\nFINAL ALWAYS-ON RESULTS')
print('-' * 60)
display(target_df)
print('ALL THREE 85% TARGETS MET:', all_targets_met)
print('Selected candidate:', selected_candidate)
print('Saved to:', RUN_DIR)
display(percentage_summary_df)
display(category_summary_df)

## Drive Outputs

Everything is saved under:

`/content/drive/MyDrive/Thesis/Week 3/results/week3_always_on_lora_sweep_v1`

- `candidate_models/<candidate>/adapter`: best adapter from every configuration.
- `final_selected_adapter`: convenient copy of the global winner.
- `results/all_candidate_selection_metrics.csv`
- `results/candidate_best_summary.csv`
- `results/base_synthetic_results.csv`
- `results/base_general_control_results.csv`
- `results/base_general_validation_results.csv`
- `results/final_lora_synthetic_results.csv`
- `results/final_lora_general_control_results.csv`
- `results/all_final_test_results.csv`
- `results/before_after_comparison.csv`
- `results/percentage_summary.csv`
- `results/percentage_by_category.csv`
- `results/target_achievement.csv`
- `results/candidate_configs.json`
- `results/metrics.json`

The final general-control and synthetic evaluation sets are not used for
training or model selection.